In [1]:
import os
import sys

sys.path.append("../../../../")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import copy
import torch
from datetime import datetime
from torch.utils.data import DataLoader
from utils.helper import ModelConfig, color_print
from utils.dataset_utils.load_dataset import (
    load_data,
)
from utils.model_utils.save_module import save_module
from utils.model_utils.load_model import load_model
from utils.model_utils.evaluate import evaluate_model, get_sparsity, similar
from utils.dataset_utils.perturb import make_example
from utils.dataset_utils.sampling import SamplingDataset
from utils.prune_utils.prune_head import head_importance_prunning
from utils.prune_utils.prune import prune_concern_identification

In [3]:
name = "bert-6-128-yahoo"
device = torch.device("cuda:0")
checkpoint = None
batch_size = 16
num_workers = 4
num_samples = 128
head_pruning_ratio = 0.3
ci_ratio = 0.3
seed = 44
include_layers = ["intermediate", "output"]
exclude_layers = ["attention"]

In [4]:
script_start_time = datetime.now()
print(f"Script started at: {script_start_time.strftime('%Y-%m-%d %H:%M:%S')}")

Script started at: 2024-09-13 15:38:40


In [5]:
model_config = ModelConfig(name, device)
num_labels = model_config.config["num_labels"]
model, tokenizer, checkpoint = load_model(model_config)

Loading the model.

{'model_name': 'Models/bert-6-128-yahoo', 'tokenizer_name': 'sadickam/sdg-classification-bert', 'task_type': 'classification', 'architectures': 'bert', 'dataset_name': 'YahooAnswersTopics', 'num_labels': 10, 'cache_dir': 'Models'}

The model Models/bert-6-128-yahoo is loaded.

In [6]:
train_dataloader, valid_dataloader, test_dataloader = load_data(
    model_config.dataset_name,
    batch_size=batch_size,
    num_workers=num_workers,
    do_cache=True,
    seed=seed,
)

Loading cached dataset YahooAnswersTopics.

train.pkl is loaded from cache.

valid.pkl is loaded from cache.

test.pkl is loaded from cache.

The dataset YahooAnswersTopics is loaded

{'dataset_name': 'YahooAnswersTopics', 'path': 'yahoo_answers_topics', 'config_name': 'yahoo_answers_topics', 'text_column': 'question_title', 'label_column': 'topic', 'cache_dir': 'Datasets/YahooAnswersTopics', 'task_type': 'classification'}

In [7]:
positive_embeddings, negative_embeddings = make_example(
    model,
    model_config,
    data_loader=train_dataloader,
    example_num=3000,
    top_emb=1,
    bottom_emb=0,
    true_ratio=0.5,
    step_eps=0.01,
    max_eps=10.0,
)

class 0

class 1

class 2

class 3

class 4

class 5

class 6

class 7

class 8

class 9

In [8]:
# from utils.dataset_utils.load_dataset import save_cache, load_from_cache
# save_cache(positive_embeddings, "./", "positive_embeddings.pkl")
# save_cache(negative_embeddings, "./", "negative_embeddings.pkl")

In [9]:
# from utils.dataset_utils.load_dataset import save_cache, load_from_cache
# positive_embeddings = load_from_cache("./", "positive_embeddings.pkl")
# negative_embeddings = load_from_cache("./", "negative_embeddings.pkl")

In [10]:
pos_dataloader = DataLoader(
    positive_embeddings, batch_size=batch_size, shuffle=True, num_workers=0
)
neg_dataloader = DataLoader(
    negative_embeddings, batch_size=batch_size, shuffle=True, num_workers=0
)

In [11]:
# print("Evaluate the original model")
# result = evaluate_model(model, model_config, test_dataloader)

In [12]:
for concern in range(num_labels):
    valid = copy.deepcopy(valid_dataloader)

    module = copy.deepcopy(model)
    pos = copy.deepcopy(pos_dataloader)
    neg = copy.deepcopy(neg_dataloader)

    positive_samples = SamplingDataset(
        pos,
        concern,
        num_samples,
        num_labels,
        True,
        4,
        device=device,
        resample=False,
        seed=seed,
    )
    negative_samples = SamplingDataset(
        neg,
        concern,
        num_samples,
        num_labels,
        False,
        4,
        device=device,
        resample=False,
        seed=seed,
    )
    all_samples = SamplingDataset(
        pos,
        200,
        num_samples,
        num_labels,
        False,
        4,
        device=device,
        resample=False,
        seed=seed,
    )

    head_importance_prunning(module, model_config, positive_samples, head_pruning_ratio)

    prune_concern_identification(
        module,
        model_config,
        positive_samples,
        negative_samples,
        include_layers=include_layers,
        exclude_layers=exclude_layers,
        sparsity_ratio=ci_ratio,
    )

    print(f"Evaluate the pruned model {concern}")
    result = evaluate_model(module, model_config, test_dataloader)
    get_sparsity(module)

    similar(
        model, module, valid, concern, num_samples, num_labels, device=device, seed=seed
    )

    save_module(
        module,
        "Modules/",
        f"head_pruned_ci_{name}_{head_pruning_ratio}p_{ci_ratio}p_class{concern}",
    )

Total heads to prune: 2

BertSdpaSelfAttention is used but `torch.nn.functional.scaled_dot_product_attention` does not support non-absolute `position_embedding_type` or `output_attentions=True` or `head_mask`. Falling back to the manual attention implementation, but specifying the manual implementation will be required from Transformers version v5.0.0 onwards. This warning can be removed using the argument `attn_implementation="eager"` when loading the model.


{(3, 1), (2, 1)}

Evaluate the pruned model 0

Evaluating the model:   0%|                                                                    | 0/1875 [00:00…

...

CCA coefficients mean non-concern: (0.9927306232255426, 0.9927306232255426)

Linear CKA concern: 0.9670715399759664

Linear CKA non-concern: 0.9763057839392242

Kernel CKA concern: 0.9477084912615894

Kernel CKA non-concern: 0.9590672195114626